In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SmartHomeEnergyTracker") \
    .getOrCreate()

data = [
    (101, "Living Room", "2026-06-01 08:00:00", 2.5),
    (101, "Living Room", "2026-06-01 19:00:00", 3.2),
    (102, "Kitchen", "2026-06-01 10:00:00", 4.0),
    (102, "Kitchen", "2026-06-01 20:00:00", 5.1),
    (103, "Bedroom", "2026-06-01 14:00:00", 1.8),
    (103, "Bedroom", "2026-06-01 21:00:00", 2.7),
    (104, "Office", "2026-06-01 09:00:00", 6.5),
    (104, "Office", "2026-06-01 18:30:00", 7.2),
    (105, "Garage", "2026-06-01 11:00:00", 3.6),
    (105, "Garage", "2026-06-01 22:00:00", 4.4)
]

columns = [
    "device_id",
    "room",
    "timestamp",
    "energy_kwh"
]

df = spark.createDataFrame(data, columns)

df.show()

+---------+-----------+-------------------+----------+
|device_id|       room|          timestamp|energy_kwh|
+---------+-----------+-------------------+----------+
|      101|Living Room|2026-06-01 08:00:00|       2.5|
|      101|Living Room|2026-06-01 19:00:00|       3.2|
|      102|    Kitchen|2026-06-01 10:00:00|       4.0|
|      102|    Kitchen|2026-06-01 20:00:00|       5.1|
|      103|    Bedroom|2026-06-01 14:00:00|       1.8|
|      103|    Bedroom|2026-06-01 21:00:00|       2.7|
|      104|     Office|2026-06-01 09:00:00|       6.5|
|      104|     Office|2026-06-01 18:30:00|       7.2|
|      105|     Garage|2026-06-01 11:00:00|       3.6|
|      105|     Garage|2026-06-01 22:00:00|       4.4|
+---------+-----------+-------------------+----------+



In [2]:
from pyspark.sql.functions import *

df = df.withColumn(
    "timestamp",
    to_timestamp("timestamp")
)

df = df.withColumn(
    "hour",
    hour("timestamp")
)

df = df.withColumn(
    "usage_type",
    when(
        (col("hour") >= 18) &
        (col("hour") <= 22),
        "Peak"
    ).otherwise("Off-Peak")
)

peak_usage = df.groupBy(
    "device_id",
    "usage_type"
).agg(
    sum("energy_kwh").alias("total_energy")
)

peak_usage.show()

+---------+----------+------------+
|device_id|usage_type|total_energy|
+---------+----------+------------+
|      101|  Off-Peak|         2.5|
|      102|      Peak|         5.1|
|      101|      Peak|         3.2|
|      103|  Off-Peak|         1.8|
|      102|  Off-Peak|         4.0|
|      104|      Peak|         7.2|
|      104|  Off-Peak|         6.5|
|      105|      Peak|         4.4|
|      105|  Off-Peak|         3.6|
|      103|      Peak|         2.7|
+---------+----------+------------+



In [3]:
top_devices = df.groupBy(
    "device_id"
).agg(
    sum("energy_kwh").alias("total_usage")
).orderBy(
    col("total_usage").desc()
)

top_devices.show()

+---------+-----------+
|device_id|total_usage|
+---------+-----------+
|      104|       13.7|
|      102|        9.1|
|      105|        8.0|
|      101|        5.7|
|      103|        4.5|
+---------+-----------+



In [4]:
top_devices.write \
    .mode("overwrite") \
    .csv("/tmp/top_energy_devices")